In [1]:
import pandas as pd
import numpy as np

features = pd.read_csv("../outputs/feature_table.csv")

print("Dataset shape:", features.shape)
print("Columns:", list(features.columns))
features.head()

Dataset shape: (95, 12)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility']


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Institute_Count,Institutional_Gap_Score,Company_Spread,Company_Spread_Norm,Velocity,Volatility
0,advanced sql,8,-4.000000,0.812500,1,1,3,0.4,8,0.8,-4.0,2.607681
1,api gateway,1,0.000000,0.250000,0,0,0,1.0,1,0.1,0.0,0.447214
2,api rate limiting,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214
3,auto scaling,1,0.000000,0.500000,0,0,0,1.0,1,0.1,0.0,0.447214
4,aws,29,-1.571429,0.318966,0,1,5,0.0,10,1.0,-2.0,4.494441


In [ ]:
features["Demand_Score"] = (
    features["Freq"]                 * 0.25 +
    features["Trend_Slope"]          * 0.20 +
    features["Recency_Weight"]       * 0.20 +
    features["Velocity"]             * 0.15 +
    features["Company_Spread_Norm"]  * 0.20
).round(4)

threshold = features["Demand_Score"].quantile(0.65)
print(f"Demand_Score threshold (65th percentile): {threshold:.4f}")

features["Label_MSRIT"] = (
    (features["Demand_Score"] > threshold) &
    (features["Is_Taught_MSRIT"] == 0)
).astype(int)

print("Class distribution:")
print(features["Label_MSRIT"].value_counts())
print(f"Imbalance ratio: {features['Label_MSRIT'].value_counts()[0] / features['Label_MSRIT'].value_counts()[1]:.1f}:1")
print()
print("Demand_Score stats:")
print(features["Demand_Score"].describe().round(3))

Demand_Score threshold (65th percentile): 1.6223
Class distribution:
Label_MSRIT
0    78
1    17
Name: count, dtype: int64
Imbalance ratio: 4.6:1

Demand_Score stats:
count    95.000
mean      1.859
std       2.386
min       0.270
25%       0.370
50%       0.470
75%       2.717
max      12.074
Name: Demand_Score, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    "Freq",
    "Trend_Slope",
    "Recency_Weight",
    "Upcoming_Flag",
    "Taught_Institute_Count",
    "Company_Spread",
    "Company_Spread_Norm",
    "Velocity",
    "Volatility",
    "Institutional_Gap_Score",
]

missing_cols = [c for c in FEATURE_COLS if c not in features.columns]
if missing_cols:
    print("WARNING - missing columns:", missing_cols)
else:
    print("All feature columns present")

X = features[FEATURE_COLS]
y = features["Label_MSRIT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])
print("Train class dist:", y_train.value_counts().to_dict())
print("Test class dist:", y_test.value_counts().to_dict())

All feature columns present
Train size: 71 | Test size: 24
Train class dist: {0: 58, 1: 13}
Test class dist: {0: 20, 1: 4}


In [4]:
try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=42, k_neighbors=3)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    print("After SMOTE:")
    print("Train size:", X_train_bal.shape[0])
    print("Class distribution:", pd.Series(y_train_bal).value_counts().to_dict())
    smote_available = True

except ImportError:
    print("imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("Falling back to class_weight='balanced' only.")
    X_train_bal, y_train_bal = X_train, y_train
    smote_available = False

After SMOTE:
Train size: 116
Class distribution: {1: 58, 0: 58}


In [ ]:
%pip install xgboost

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)
rf_model.fit(X_train_bal, y_train_bal)
print("Random Forest trained on", len(FEATURE_COLS), "features")

try:
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=y_train_bal.value_counts()[0] / y_train_bal.value_counts()[1],
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    xgb_model.fit(X_train_bal, y_train_bal)
    print("XGBoost trained")
    xgb_available = True
except Exception as e:
    print(f"XGBoost unavailable: {e}")
    xgb_available = False

model = rf_model

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Random Forest trained on 10 features
XGBoost trained


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

probs_test = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.20, 0.70, 0.05)
results = []
for t in thresholds:
    preds = (probs_test > t).astype(int)
    results.append({
        "Threshold": round(t, 2),
        "F1_gap":        round(f1_score(y_test, preds, zero_division=0), 3),
        "Precision_gap": round(precision_score(y_test, preds, zero_division=0), 3),
        "Recall_gap":    round(recall_score(y_test, preds, zero_division=0), 3),
    })

threshold_df = pd.DataFrame(results)
best_row = threshold_df.loc[threshold_df["F1_gap"].idxmax()]
best_threshold = best_row["Threshold"]

print("Random Forest threshold tuning:")
print(threshold_df.to_string(index=False))
print(f"Best threshold (RF): {best_threshold}")

if xgb_available:
    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    xgb_results = []
    for t in thresholds:
        preds = (xgb_probs > t).astype(int)
        xgb_results.append({
            "Threshold": round(t, 2),
            "F1_gap":        round(f1_score(y_test, preds, zero_division=0), 3),
            "Precision_gap": round(precision_score(y_test, preds, zero_division=0), 3),
            "Recall_gap":    round(recall_score(y_test, preds, zero_division=0), 3),
        })
    xgb_threshold_df = pd.DataFrame(xgb_results)
    best_xgb_row = xgb_threshold_df.loc[xgb_threshold_df["F1_gap"].idxmax()]
    print(f"XGBoost best F1: {best_xgb_row['F1_gap']} at threshold {best_xgb_row['Threshold']}")
    print(f"RFbest F1: {best_row['F1_gap']} at threshold {best_threshold}")
    winner = "XGBoost" if best_xgb_row["F1_gap"] > best_row["F1_gap"] else "Random Forest"
    print(f"Winner: {winner}")

Random Forest threshold tuning:
 Threshold  F1_gap  Precision_gap  Recall_gap
      0.20   0.571          0.400        1.00
      0.25   0.727          0.571        1.00
      0.30   0.800          0.667        1.00
      0.35   0.667          0.600        0.75
      0.40   0.750          0.750        0.75
      0.45   0.750          0.750        0.75
      0.50   0.750          0.750        0.75
      0.55   0.750          0.750        0.75
      0.60   0.750          0.750        0.75
      0.65   0.750          0.750        0.75
Best threshold (RF): 0.3
XGBoost best F1: 0.667 at threshold 0.2
RFbest F1: 0.8 at threshold 0.3
Winner: Random Forest


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = (probs_test > best_threshold).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Not a Gap", "Gap Skill"]))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
    index=["Actual: Not Gap", "Actual: Gap"],
    columns=["Predicted: Not Gap", "Predicted: Gap"]
)
print(cm_df)

roc_auc = roc_auc_score(y_test, probs_test)
print(f"\nROC-AUC Score: {roc_auc:.3f}")
print("(ROC-AUC > 0.75 is good for imbalanced binary classification)")

Classification Report:
              precision    recall  f1-score   support

   Not a Gap       1.00      0.90      0.95        20
   Gap Skill       0.67      1.00      0.80         4

    accuracy                           0.92        24
   macro avg       0.83      0.95      0.87        24
weighted avg       0.94      0.92      0.92        24

Confusion Matrix:
                 Predicted: Not Gap  Predicted: Gap
Actual: Not Gap                  18               2
Actual: Gap                       0               4

ROC-AUC Score: 0.938
(ROC-AUC > 0.75 is good for imbalanced binary classification)


In [ ]:
importance = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.to_string(index=False))

                Feature  Importance
             Volatility    0.264860
                   Freq    0.199451
    Company_Spread_Norm    0.140255
         Company_Spread    0.125975
Institutional_Gap_Score    0.097292
 Taught_Institute_Count    0.070771
               Velocity    0.051459
            Trend_Slope    0.030854
         Recency_Weight    0.014206
          Upcoming_Flag    0.004877


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = cross_val_predict(
    RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=2,
        random_state=42, class_weight="balanced"
    ),
    X, y,
    cv=cv,
    method="predict_proba"
)[:, 1]

features["Recommendation_Prob_MSRIT"] = oof_probs
print("Out-of-fold probabilities computed — no data leakage")
print("Prob range:", oof_probs.min().round(3), "to", oof_probs.max().round(3))
print("Mean prob:", oof_probs.mean().round(3))

Out-of-fold probabilities computed — no data leakage
Prob range: 0.0 to 0.885
Mean prob: 0.201


In [ ]:
industry = pd.read_csv("../outputs/cleaned_industry.csv")
skill_type = industry.groupby("Skill")["Skill_Type"].agg(lambda x: x.mode()[0]).reset_index()
skill_type.columns = ["Skill", "Trajectory"]

features = features.merge(skill_type, on="Skill", how="left")
features["Trajectory"] = features["Trajectory"].fillna("Unknown")

print("Trajectory distribution:")
print(features["Trajectory"].value_counts())

Trajectory distribution:
Trajectory
Transitional    51
Upcoming        26
Traditional     18
Name: count, dtype: int64


In [11]:
gap_skills = features[features["Is_Taught_MSRIT"] == 0].copy()

ranked_msrit = gap_skills.sort_values(
    "Recommendation_Prob_MSRIT", ascending=False
).reset_index(drop=True)

display_cols = [
    "Skill", "Trajectory", "Freq", "Velocity", "Volatility",
    "Company_Spread", "Recency_Weight", "Upcoming_Flag",
    "Demand_Score", "Label_MSRIT", "Recommendation_Prob_MSRIT"
]

print("Top 20 Recommended Skills for MSRIT Curriculum:")
print(ranked_msrit[display_cols].head(20).to_string(index=False))

Top 20 Recommended Skills for MSRIT Curriculum:
              Skill   Trajectory  Freq  Velocity  Volatility  Company_Spread  Recency_Weight  Upcoming_Flag  Demand_Score  Label_MSRIT  Recommendation_Prob_MSRIT
             docker Transitional    13  0.000000    5.813777               8        0.250000              0        3.4600            1                   0.884969
              linux Transitional    11  0.000000    4.919350               8        0.000000              0        2.9100            1                   0.866668
      generative ai     Upcoming    14 -8.000000    4.764452               9        0.803571              1        1.0407            0                   0.855932
          terraform  Traditional    17 -0.333333    2.607681               9        0.602941              1        4.4406            1                   0.847684
             golang Transitional    22 -2.500000    4.929503               9        0.693182              0        4.9436            1        

In [ ]:
ranked_msrit.to_csv("../outputs/ranked_skills_msrit.csv", index=False)
print("Saved ranked_skills_msrit.csv — shape:", ranked_msrit.shape)

features.to_csv("../outputs/model_output.csv", index=False)
print("Saved model_output.csv — shape:", features.shape)
print("Columns:", list(features.columns))

print("Top 10 gap skills for MSRIT:")
print(ranked_msrit[["Skill","Trajectory","Recommendation_Prob_MSRIT","Demand_Score","Company_Spread"]].head(10).to_string(index=False))

Saved ranked_skills_msrit.csv — shape: (73, 16)
Saved model_output.csv — shape: (95, 16)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Demand_Score', 'Label_MSRIT', 'Recommendation_Prob_MSRIT', 'Trajectory']
Top 10 gap skills for MSRIT:
        Skill   Trajectory  Recommendation_Prob_MSRIT  Demand_Score  Company_Spread
       docker Transitional                   0.884969        3.4600               8
        linux Transitional                   0.866668        2.9100               8
generative ai     Upcoming                   0.855932        1.0407               9
    terraform  Traditional                   0.847684        4.4406               9
       golang Transitional                   0.842729        4.9436               9
          css Transitional                   0.814307        1.8982               9
   k